# LABORATORIO N.° 03 — La métrica que importa

**Curso:** Analítica Empresarial Integrada  
**Semana 3:** KPI accionables, North Star Metric y árbol de métricas  
**Docente:** Pilar Rocío Sayán Mejía  
**Periodo:** 2026-II

**Fuente real:** UCI Machine Learning Repository — Online Retail (ID 352).

> Objetivo del notebook: conectar **objetivo → North Star → drivers → guardrails → KPI → tablero → interpretación → decisión** usando transacciones reales.
---

**Apellidos y nombres del estudiante:** _Cuya Vera Juandiego Alejandro_

---


## Agenda de laboratorio — 7:00 p. m. a 10:10 p. m.

**Duración total:** 190 minutos · **Receso:** 15 minutos · **Trabajo efectivo:** 175 minutos.

| Horario | Tiempo | Desarrollo |
|---|---:|---|
| 7:00–7:10 | 10 min | Apertura del caso y presentación del problema de medición. |
| 7:10–7:35 | 25 min | Actividad 1. Revisión de conceptos: del dato a la decisión. |
| 7:35–8:00 | 25 min | Actividad 2. Descarga desde UCI, auditoría y limpieza documentada. |
| 8:00–8:30 | 30 min | Actividad 2. Periodo comparable, recurrencia y North Star. Reto 1. |
| 8:30–8:45 | 15 min | **RECESO** |
| 8:45–9:15 | 30 min | Actividad 2. Árbol de métricas, guardrails, Polars y DuckDB. |
| 9:15–9:35 | 20 min | Actividad 2. Tablero de decisión en Plotly. Reto 2. |
| 9:35–10:00 | 25 min | **Reto de aplicación y retroalimentación.** Ejercicios 1 a 5. |
| 10:00–10:10 | 10 min | Diccionario de KPI, informe ejecutivo y ticket de salida. |

> **Regla de trabajo:** no avance de bloque sin registrar la interpretación solicitada. El objetivo no es ejecutar celdas, sino convertir datos en evidencia para una decisión.


## Actividad 1 — Revisión de conceptos: del dato a la decisión (25 minutos)

**Propósito.** Establecer con precisión el vocabulario de medición antes de programar. La confusión entre dato, métrica, indicador y KPI constituye la causa más frecuente de tableros que no sustentan ninguna decisión.

**Instrucciones.** Complete la tabla con definiciones elaboradas con sus propias palabras. No se admite la reproducción literal de fuentes externas ni de sistemas generativos. La columna de la derecha contiene una pregunta de apoyo: si su definición permite responderla, la definición es suficiente; si no lo permite, corríjala antes de continuar.

**Evidencia esperada.** Tabla completa con las definiciones registradas.

| **Concepto** | **Definición elaborada por el estudiante** | **Pregunta de apoyo** |
|---|---|---|
| **Dato** | Es un registro puntual que describe un hecho, valor o característica. Por sí mismo no siempre brinda información suficiente para decidir. **Ejemplo:** registrar una venta de US$ 5000 en la base de datos. | ¿En qué se diferencia un dato de una métrica? |
| **Métrica** | Es una medida que se obtiene al procesar, resumir o calcular datos para cuantificar algún aspecto del negocio. Una métrica puede estar bien calculada y aun así no ser útil para una decisión si no está vinculada con un objetivo. | ¿Toda métrica calculada correctamente resulta útil para decidir? |
| **Indicador** | Es una métrica interpretada dentro de un contexto o comparada con algún criterio, de modo que permita reconocer si la situación es adecuada, desfavorable o requiere atención. | ¿Qué debe añadirse a una métrica para que constituya un indicador? |
| **KPI** | Es un indicador clave que se prioriza porque está relacionado directamente con un objetivo relevante de la organización y permite hacer seguimiento a su avance. | ¿Por qué una organización no puede sostener veinte KPI simultáneos? |
| **Meta** | Es el valor objetivo definido como referencia para contrastarlo posteriormente con el resultado realmente alcanzado. | ¿Qué distingue un valor observado en la base de una meta propuesta para el ejercicio? |
| **North Star** | Es la métrica central que resume el valor que la organización pretende generar para sus clientes y sirve como guía para orientar decisiones. Para evitar que sea una métrica de vanidad, debe representar valor real. | ¿Qué condición debe cumplir una North Star para no convertirse en métrica de vanidad? |
| **Driver** | Es una variable relacionada con el movimiento de la North Star. Para considerarla como driver, debe revisarse si sus cambios mantienen una relación consistente con las variaciones de la métrica principal. | ¿Cómo se comprueba que una variable es efectivamente driver de la North Star? |
| **Guardrail** | Es una métrica de control utilizada para vigilar límites o posibles efectos no deseados mientras se busca mejorar la North Star. | ¿Qué ocurre si una organización optimiza su North Star sin vigilar los guardrails? |

## Criterio de cierre

Las definiciones permiten distinguir los datos de las métricas calculadas y muestran cómo cada indicador adquiere sentido cuando se relaciona con una decisión empresarial.


## Actividad 2 — Desarrollo práctico y ejecución

### Presentación del caso

Una empresa minorista en línea del Reino Unido cuenta con un registro histórico de transacciones que incluye facturas, productos, cantidades, fechas, precios unitarios, clientes y países. La dirección necesita transformar estos registros en un sistema de métricas que permita distinguir crecimiento útil de crecimiento aparente. Para ello, no basta con observar ventas acumuladas o cantidad de clientes: es necesario identificar qué indicadores representan valor recurrente para el cliente y cuáles pueden orientar una decisión empresarial.

Durante el laboratorio, el equipo trabajará con el conjunto Online Retail del UCI Machine Learning Repository. A partir de los datos reales, deberá auditar y preparar las transacciones, identificar clientes recurrentes, proponer y justificar una North Star, descomponerla en drivers y guardrails, construir un diccionario de KPI y elaborar un tablero de decisión en Plotly. El análisis deberá terminar con hallazgos cuantitativos y una acción empresarial concreta, diferenciando en todo momento los valores observados en la base de las metas o umbrales académicos propuestos para el ejercicio.


## Paso 1 — Preparación reproducible del entorno


In [ ]:
# Instalamos las versiones necesarias para mantener un entorno reproducible
%pip install -q ucimlrepo==0.0.7 polars==1.17.1 duckdb==1.1.3

# Importamos las librerías para trabajar con datos, consultas y gráficos
import polars as pl
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from ucimlrepo import fetch_ucirepo

# Configuramos la cantidad de filas y el ancho de las tablas mostradas
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(160)

# Definimos el formato real de las fechas del conjunto de datos
FORMATO_FECHA = ["%m/%d/%Y %H:%M", "%d/%m/%Y %H:%M"]

# Verificamos las versiones instaladas y que el entorno esté listo
print("polars:", pl.__version__, "| duckdb:", duckdb.__version__)
print("Entorno listo.")

polars: 1.17.1 | duckdb: 1.1.3
Entorno listo.


> **Interpretación:** En esta celda se deja preparado un entorno reproducible para trabajar con el conjunto Online Retail. Se instalan las versiones requeridas, se importan las librerías necesarias para procesar los datos, consultar información y generar gráficos, y también se define el formato de las fechas para prevenir problemas durante su conversión.

---


## Paso 2 — Descarga de datos reales desde UCI

El conjunto **Online Retail** contiene transacciones de una empresa minorista en línea registrada en el Reino Unido entre diciembre de 2010 y diciembre de 2011. Los códigos de factura que empiezan con `C` representan cancelaciones. No se generan registros artificiales.


In [ ]:
# Descargamos el conjunto de datos Online Retail desde el repositorio UCI
online_retail = fetch_ucirepo(id=352)

# Convertimos los datos descargados de pandas a un DataFrame de Polars
df = pl.from_pandas(online_retail.data.original)

# Mostramos el nombre del conjunto de datos
print("Dataset:", online_retail.metadata.get("name"))

# Mostramos la cantidad de filas y columnas
print("Filas y columnas:", df.shape)

# Mostramos los nombres de las columnas disponibles
print("Columnas:", df.columns)

# Visualizamos las primeras cinco filas para revisar la estructura
display(df.head())

Dataset: Online Retail
Filas y columnas: (541909, 8)
Columnas: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
str,str,str,i64,str,f64,f64,str
"""536365""","""85123A""","""WHITE HANGING HEART T-LIGHT HO…",6,"""12/1/2010 8:26""",2.55,17850.0,"""United Kingdom"""
"""536365""","""71053""","""WHITE METAL LANTERN""",6,"""12/1/2010 8:26""",3.39,17850.0,"""United Kingdom"""
"""536365""","""84406B""","""CREAM CUPID HEARTS COAT HANGER""",8,"""12/1/2010 8:26""",2.75,17850.0,"""United Kingdom"""
"""536365""","""84029G""","""KNITTED UNION FLAG HOT WATER B…",6,"""12/1/2010 8:26""",3.39,17850.0,"""United Kingdom"""
"""536365""","""84029E""","""RED WOOLLY HOTTIE WHITE HEART.""",6,"""12/1/2010 8:26""",3.39,17850.0,"""United Kingdom"""


> **Interpretación:** El conjunto Online Retail se obtiene directamente desde UCI y luego se transforma a un DataFrame de Polars para continuar con el procesamiento. La base contiene 541 909 registros distribuidos en 8 columnas, con datos relacionados con facturas, productos, cantidades, fechas, precios, clientes y países.


**Control de trazabilidad**

**Indicadores del dataset:**

- **Filas:** 541,909
- **Facturas:** 25,900
- **Clientes:** 4,372
- **Cancelaciones:** 9,288
- **Cantidades no positivas:** 10,624
- **Precios no positivos:** 2,517

**Países presentes:** 38

---

## Paso 3 — Auditoría inicial y limpieza documentada


In [ ]:
# Convertimos cada columna al tipo de dato que necesitamos para trabajar
df = df.with_columns([
    # InvoiceNo se maneja como texto porque contiene números y códigos como "C"
    pl.col("InvoiceNo").cast(pl.Utf8),

    # CustomerID se maneja como texto para conservar correctamente el identificador
    pl.col("CustomerID").cast(pl.Utf8),

    # Quantity se convierte a número decimal; los valores no convertibles pasan a nulo
    pl.col("Quantity").cast(pl.Float64, strict=False),

    # UnitPrice se convierte a número decimal; los valores no convertibles pasan a nulo
    pl.col("UnitPrice").cast(pl.Float64, strict=False),

    # InvoiceDate se convierte primero a texto para aplicar el formato de fecha definido
    pl.col("InvoiceDate").cast(pl.Utf8)
      .str.to_datetime(format=FORMATO_FECHA, strict=False)
      .alias("InvoiceDate"),
])

# Creamos variables derivadas para identificar cancelaciones y calcular el importe
df = df.with_columns([
    # Una factura cuyo código comienza con "C" se considera una cancelación
    pl.col("InvoiceNo")
      .str.to_uppercase()
      .str.starts_with("C")
      .alias("es_cancelacion"),

    # Calculamos el importe de cada línea multiplicando cantidad por precio unitario
    (pl.col("Quantity") * pl.col("UnitPrice")).alias("importe_linea"),
])

# Contamos las fechas que no pudieron convertirse al formato definido
nulas = df["InvoiceDate"].null_count()

# Mostramos el resultado del control de fechas
print("Fechas que no pudieron parsearse:", nulas)

# Si existen fechas nulas, detenemos el proceso porque el formato podría ser incorrecto
if nulas:
    raise ValueError("El formato declarado en FORMATO_FECHA no corresponde a la fuente.")

# Definimos los valores esperados para comprobar que la fuente no haya cambiado
FILAS_ESPERADAS = 541_909
FACTURAS_ESPERADAS = 25_900

# Verificamos que la cantidad de filas coincida con la fuente original
if df.height != FILAS_ESPERADAS:
    raise ValueError(
        f"La fuente cambio: se esperaban {FILAS_ESPERADAS:,} filas y llegaron {df.height:,}. "
        "Avise al docente antes de continuar."
    )

# Verificamos que la cantidad de facturas distintas también coincida
if df["InvoiceNo"].n_unique() != FACTURAS_ESPERADAS:
    raise ValueError(
        f"La fuente cambio: se esperaban {FACTURAS_ESPERADAS:,} facturas distintas "
        f"y llegaron {df['InvoiceNo'].n_unique():,}."
    )

# Confirmamos que la fuente mantiene la cantidad de datos esperada
print(
    "Integridad verificada:",
    f"{df.height:,}",
    "filas y",
    f"{df['InvoiceNo'].n_unique():,}",
    "facturas, como se esperaba."
)

# Construimos una tabla resumen con los principales controles de calidad
control = pl.DataFrame({
    # Definimos los nombres de los indicadores que vamos a revisar
    "indicador": [
        "filas",
        "facturas",
        "clientes",
        "cancelaciones",
        "cantidades_no_positivas",
        "precios_no_positivos"
    ],

    # Calculamos el valor correspondiente a cada indicador
    "valor": [
        df.height,
        df["InvoiceNo"].n_unique(),
        df["CustomerID"].drop_nulls().n_unique(),
        int(df["es_cancelacion"].sum()),
        int((df["Quantity"] <= 0).sum()),
        int((df["UnitPrice"] <= 0).sum())
    ],
})

# Mostramos la tabla de control
display(control)

# Mostramos la primera y última fecha registrada en el dataset
print("Rango:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())

Fechas que no pudieron parsearse: 0
Integridad verificada: 541,909 filas y 25,900 facturas, como se esperaba.


indicador,valor
str,i64
"""filas""",541909
"""facturas""",25900
"""clientes""",4372
"""cancelaciones""",9288
"""cantidades_no_positivas""",10624
"""precios_no_positivos""",2517


Rango: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00


In [20]:
# Filtramos el DataFrame original para obtener solo compras válidas
compras = df.filter(

    # Excluimos las facturas que corresponden a cancelaciones
    (~pl.col("es_cancelacion"))

    # La cantidad comprada debe ser mayor que cero
    & (pl.col("Quantity") > 0)

    # El precio unitario debe ser mayor que cero
    & (pl.col("UnitPrice") > 0)

    # La fecha de la factura no debe estar vacía
    & pl.col("InvoiceDate").is_not_null()

    # El cliente debe estar identificado
    & pl.col("CustomerID").is_not_null()

)

# Mostramos la cantidad de filas que tenía el dataset original
print("Filas originales:", df.height)

# Mostramos cuántas filas quedaron después del filtro
print("Filas de compra validas:", compras.height)

# Calculamos qué porcentaje de las filas originales se conservó
print("Proporcion conservada: {:.1%}".format(compras.height / df.height))

Filas originales: 541909
Filas de compra validas: 397884
Proporcion conservada: 73.4%


### Reto de calidad

**¿Qué sesgo aparece si contamos cancelaciones como ventas?**  
Se generaría una **sobrevaloración de las ventas**, ya que las operaciones canceladas terminarían sumándose como si fueran ingresos válidos, en vez de excluirse o considerarse correctamente.

**¿Por qué no debemos borrar esos registros de la fuente original?**  
Porque las cancelaciones también forman parte del **registro histórico real** y son necesarias para auditar lo sucedido. Por ello, conviene conservar intacta la fuente y realizar la limpieza en una tabla o copia derivada.


## Paso 4 — Definición del periodo comparable

La fuente empieza el 1/12/2010 y termina el 9/12/2011. Para no comparar meses incompletos con meses completos, el laboratorio informa sobre **enero–noviembre de 2011**.

**Cuidado con el sesgo de ventana.** La tabla de facturas se construye sobre *todo* el historial disponible, y el recorte a la ventana se aplica **después** de determinar la recurrencia. Si se recorta antes, un cliente que compró en diciembre de 2010 y volvió en enero de 2011 aparece como comprador nuevo, y la North Star crece de forma artificial en los primeros meses: en esta base, enero pasa de 570 a 246 compras recurrentes, un 57 % menos, solo por haber filtrado en el orden equivocado.


In [21]:
# Agregamos una columna "mes" con el año y mes de cada compra
compras = compras.with_columns(
    pl.col("InvoiceDate").dt.strftime("%Y-%m").alias("mes")
)

# Definimos el inicio del periodo comparable: 1 de enero de 2011
INICIO = pl.datetime(2011, 1, 1)

# Definimos el final del periodo: 1 de diciembre de 2011
FIN = pl.datetime(2011, 12, 1)

# Construimos la tabla de facturas usando TODO el historial disponible
facturas = (
    compras.group_by(["InvoiceNo", "CustomerID", "mes"])

    # Resumimos cada factura en una sola fila
    .agg([
        # Tomamos la primera fecha registrada de cada factura
        pl.col("InvoiceDate").min().alias("fecha_factura"),

        # Sumamos los importes de todas las líneas de la factura
        pl.col("importe_linea").sum().alias("importe_factura"),

        # Sumamos las unidades compradas en la factura
        pl.col("Quantity").sum().alias("unidades"),

        # Contamos cuántas líneas de productos tiene la factura
        pl.col("StockCode").count().alias("lineas")
    ])
)

# Seleccionamos las facturas cuya fecha está dentro de enero-noviembre de 2011
en_ventana = facturas.filter(
    (pl.col("fecha_factura") >= INICIO) &
    (pl.col("fecha_factura") < FIN)
)

# Mostramos cuántas facturas existen considerando todo el historial
print("Facturas en el historial completo :", facturas["InvoiceNo"].n_unique())

# Mostramos cuántas facturas pertenecen al periodo comparable
print("Facturas en la ventana comparable :", en_ventana["InvoiceNo"].n_unique())

# Mostramos cuántos clientes aparecen en el periodo comparable
print("Clientes en la ventana            :", en_ventana["CustomerID"].n_unique())

# Calculamos los ingresos totales de la ventana en libras esterlinas
print("Ingresos en la ventana (GBP)      :", round(en_ventana["importe_factura"].sum(), 2))

# Mostramos las primeras filas de la tabla de facturas creada
display(facturas.head())

Facturas en el historial completo : 18532
Facturas en la ventana comparable : 16354
Clientes en la ventana            : 4173
Ingresos en la ventana (GBP)      : 7820501.22


InvoiceNo,CustomerID,mes,fecha_factura,importe_factura,unidades,lineas
str,str,str,datetime[μs],f64,f64,u32
"""550283""","""18094.0""","""2011-04""",2011-04-15 14:05:00,639.48,440.0,19
"""545468""","""16571.0""","""2011-03""",2011-03-03 09:41:00,436.68,378.0,29
"""575964""","""15021.0""","""2011-11""",2011-11-13 12:31:00,212.27,69.0,51
"""555283""","""17162.0""","""2011-06""",2011-06-02 09:04:00,64.3,42.0,4
"""574468""","""18172.0""","""2011-11""",2011-11-04 11:54:00,47.6,8.0,2


## Paso 5 — Recurrencia y North Star

**Regla operativa del laboratorio:** un cliente se considera recurrente desde su segunda factura válida, contada sobre **todo el historial disponible**. Su primera compra no se reclasifica retrospectivamente.

Enero de 2011 conserva un sesgo residual, porque solo dispone de un mes previo de historial. Por eso se marca como **mes de calentamiento** y se excluye de las comparaciones de variación mensual.


In [23]:
# Ordenamos las facturas por cliente, fecha y número de factura
facturas = facturas.sort(["CustomerID", "fecha_factura", "InvoiceNo"])

# Numeramos las compras de cada cliente en orden cronológico
facturas = facturas.with_columns(
    (pl.int_range(pl.len()).over("CustomerID") + 1).alias("n_compra_cliente")
)

# Consideramos recurrente al cliente desde su segunda compra
facturas = facturas.with_columns(
    (pl.col("n_compra_cliente") >= 2).alias("es_recurrente")
)

# Recortamos al periodo comparable después de determinar la recurrencia
facturas = facturas.filter(
    (pl.col("fecha_factura") >= INICIO) &
    (pl.col("fecha_factura") < FIN)
)

# Marcamos enero como mes de calentamiento por su historial previo limitado
MES_CALENTAMIENTO = "2011-01"

# Agrupamos las facturas por mes y calculamos los indicadores generales
mensual = (
    facturas.group_by("mes")
    .agg([
        # Contamos las facturas válidas de cada mes
        pl.col("InvoiceNo").n_unique().alias("facturas_validas"),

        # Contamos los clientes activos de cada mes
        pl.col("CustomerID").n_unique().alias("clientes_activos"),

        # Sumamos los ingresos de cada mes
        pl.col("importe_factura").sum().alias("ingresos")
    ])
)

# Filtramos solo las compras realizadas por clientes recurrentes
rec = (
    facturas.filter("es_recurrente")
    .group_by("mes")
    .agg([
        # Contamos las compras realizadas por clientes recurrentes
        pl.col("InvoiceNo").n_unique().alias("compras_recurrentes"),

        # Contamos los clientes recurrentes de cada mes
        pl.col("CustomerID").n_unique().alias("clientes_recurrentes"),

        # Sumamos los ingresos generados por clientes recurrentes
        pl.col("importe_factura").sum().alias("ingresos_recurrentes")
    ])
)

# Unimos los indicadores generales y de recurrencia por mes
kpi = mensual.join(rec, on="mes", how="left").fill_null(0).sort("mes")

# Calculamos el promedio de compras recurrentes por cliente recurrente
kpi = kpi.with_columns([
    (pl.col("compras_recurrentes") / pl.col("clientes_recurrentes"))
    .alias("frecuencia_recurrente"),

    # Calculamos qué porcentaje de ingresos proviene de compras recurrentes
    (100 * pl.col("ingresos_recurrentes") / pl.col("ingresos"))
    .alias("participacion_ingreso_recurrente_pct"),
])

# Mostramos la tabla final de indicadores mensuales
display(kpi)

mes,facturas_validas,clientes_activos,ingresos,compras_recurrentes,clientes_recurrentes,ingresos_recurrentes,frecuencia_recurrente,participacion_ingreso_recurrente_pct
str,u32,u32,f64,u32,u32,f64,f64,f64
"""2011-01""",987,741,569445.04,246,149,112384.24,1.651007,19.735748
"""2011-02""",997,758,447137.35,501,315,238511.25,1.590476,53.341831
"""2011-03""",1321,974,595500.76,782,486,379961.42,1.609053,63.805363
"""2011-04""",1149,856,469200.361,788,533,334553.92,1.478424,71.302997
"""2011-05""",1555,1056,678594.56,1236,779,554786.46,1.58665,81.755218
"""2011-06""",1393,991,661213.69,1125,748,562894.47,1.504011,85.130492
"""2011-07""",1331,949,600091.011,1118,761,525007.8,1.46912,87.488029
"""2011-08""",1280,935,645343.9,1095,766,562829.29,1.429504,87.213855
"""2011-09""",1755,1266,952838.382,1439,980,797153.641,1.468367,83.66095


### North Star propuesta
**Compras válidas de clientes recurrentes por mes.**

Justifique con los cuatro criterios: valor para el cliente, vínculo con valor empresarial, capacidad de influencia del equipo y descomposición en drivers.

**Respuesta:**

- Valor para el cliente: refleja que una parte de los clientes percibe suficiente utilidad en la oferta como para realizar nuevas compras.

- Valor empresarial: las compras repetidas aportan ingresos y evidencian continuidad en la relación comercial con los clientes.

- Capacidad de influencia: el equipo puede actuar sobre esta métrica mediante programas de fidelización, promociones, mejoras en la experiencia de compra y acciones de comunicación.

- Descomposición en drivers: su comportamiento puede estudiarse a partir de factores como la cantidad de clientes recurrentes, la frecuencia de compra, los productos, los países y otras variables relacionadas.


### Reto 1 — ¿Vanidad o acción?
Clasifique: productos totales, clientes activos mensuales, compras recurrentes, ingresos acumulados, tasa de cancelación y países con ventas. Para cada una indique qué decisión permite tomar.

**Respuesta:**

**Productos totales:** Se considera una **métrica de vanidad**, porque únicamente muestra la amplitud del catálogo y no permite concluir si esos productos realmente generan valor.  

**Clientes activos mensuales:** Es una métrica de **acción**, ya que ayuda a determinar en qué periodos conviene fortalecer la captación o activación de clientes.  

**Compras recurrentes:** Es una métrica de **acción**, porque permite observar el nivel de recurrencia y orientar medidas destinadas a mejorar la fidelización.  

**Ingresos acumulados:** Es una **métrica de vanidad**, debido a que informa cuánto se vendió en total, pero por sí sola no explica la calidad ni la dinámica del crecimiento.  

**Tasa de cancelación:** Es una métrica de **acción**, puesto que permite identificar el comportamiento de las cancelaciones y plantear medidas para reducirlas.  

**Países con ventas:** Es una métrica de **acción**, porque muestra qué mercados registran actividad y ayuda a decidir dónde enfocar los esfuerzos comerciales.

---


## Paso 6 — Árbol de métricas

El árbol es una **hipótesis de gestión**, no una demostración causal.

**Objetivo → North Star → drivers → guardrails**

- Objetivo: incrementar valor recurrente sin deteriorar calidad.
- North Star: compras válidas de clientes recurrentes / mes.
- Driver 1: clientes recurrentes activos.
- Driver 2: frecuencia de compra por recurrente.
- Guardrail 1: tasa de cancelación.
- Guardrail 2: concentración de ingresos en Top 10 clientes.


In [24]:
# Reconstruimos la North Star multiplicando clientes recurrentes por su frecuencia de compra
kpi = kpi.with_columns(
    (pl.col("clientes_recurrentes") * pl.col("frecuencia_recurrente"))
    .alias("ns_reconstruida")
)

# Calculamos la diferencia entre la North Star original y la reconstruida
kpi = kpi.with_columns(
    (pl.col("compras_recurrentes") - pl.col("ns_reconstruida"))
    .alias("error_reconstruccion")
)

# Mostramos los valores necesarios para comprobar la reconstrucción
display(
    kpi.select([
        "mes",
        "compras_recurrentes",
        "clientes_recurrentes",
        "frecuencia_recurrente",
        "error_reconstruccion"
    ])
)

mes,compras_recurrentes,clientes_recurrentes,frecuencia_recurrente,error_reconstruccion
str,u32,u32,f64,f64
"""2011-01""",246,149,1.651007,0.0
"""2011-02""",501,315,1.590476,5.6843e-14
"""2011-03""",782,486,1.609053,0.0
"""2011-04""",788,533,1.478424,0.0
"""2011-05""",1236,779,1.58665,0.0
"""2011-06""",1125,748,1.504011,0.0
"""2011-07""",1118,761,1.46912,0.0
"""2011-08""",1095,766,1.429504,0.0
"""2011-09""",1439,980,1.468367,0.0


**Interpretación de los resultados**

> La comprobación muestra que la **North Star puede obtenerse aritméticamente** multiplicando el número de clientes recurrentes por su frecuencia de compra.

> Para **enero**, se registraron **149 clientes recurrentes** y una frecuencia de **1.65 compras**, combinación que reproduce las **246 compras recurrentes** observadas.

> A lo largo del periodo, la frecuencia se ubica aproximadamente entre **1.43 y 1.68 compras por cliente**, y su mayor nivel se presenta en **noviembre (1.68)**.

> El **error de reconstrucción es igual a 0 o prácticamente nulo** en todos los meses. El valor `5.6843e-14` observado en febrero se explica por una diferencia mínima de precisión decimal.

> En consecuencia, los datos verifican la identidad matemática **clientes recurrentes × frecuencia = compras recurrentes**; sin embargo, esta igualdad **no constituye evidencia de causalidad**.


## Paso 7 — Guardrail 1: tasa de cancelación


In [25]:
# Filtramos los registros con fecha y cliente identificados dentro de enero-noviembre de 2011
base_total = (
    df.filter(
        pl.col("InvoiceDate").is_not_null()
        & pl.col("CustomerID").is_not_null()
        & (pl.col("InvoiceDate") >= INICIO)
        & (pl.col("InvoiceDate") < FIN)
    )
    # Creamos una columna con el año y mes de cada factura
    .with_columns(
        pl.col("InvoiceDate").dt.strftime("%Y-%m").alias("mes")
    )
)

# Conservamos una sola combinación de mes, factura, cliente y estado de cancelación
inv_total = base_total.select(
    ["mes", "InvoiceNo", "CustomerID", "es_cancelacion"]
).unique()

# Agrupamos las facturas por mes para calcular la tasa de cancelación
cancel = (
    inv_total.group_by("mes")
    .agg([
        # Contamos todas las facturas del mes
        pl.col("InvoiceNo").n_unique().alias("facturas_total"),

        # Sumamos las facturas identificadas como cancelaciones
        pl.col("es_cancelacion").sum().alias("facturas_canceladas")
    ])

    # Calculamos qué porcentaje de las facturas fueron canceladas
    .with_columns(
        (
            100 * pl.col("facturas_canceladas") /
            pl.col("facturas_total")
        ).alias("tasa_cancelacion_pct")
    )

    # Ordenamos los resultados cronológicamente
    .sort("mes")
)

# Mostramos la tabla final de cancelaciones por mes
display(cancel)

mes,facturas_total,facturas_canceladas,tasa_cancelacion_pct
str,u32,u32,f64
"""2011-01""",1236,249,20.145631
"""2011-02""",1202,204,16.971714
"""2011-03""",1619,298,18.406424
"""2011-04""",1384,235,16.979769
"""2011-05""",1849,294,15.900487
"""2011-06""",1707,314,18.394845
"""2011-07""",1593,262,16.446955
"""2011-08""",1544,263,17.033679
"""2011-09""",2078,322,15.495669


**Interpretación:**
> El procedimiento analiza las facturas comprendidas entre enero y noviembre de 2011, separa las que corresponden a cancelaciones y calcula su participación mensual. Esta tasa se utiliza como guardrail para comprobar que un aumento de las compras recurrentes no venga acompañado de un crecimiento en las cancelaciones.


## Paso 8 — Guardrail 2: concentración Top 10 clientes


In [26]:
# Agrupamos las facturas por mes y cliente
cliente_mes = (
    facturas.group_by(["mes", "CustomerID"])
    .agg(
        # Sumamos los ingresos generados por cada cliente en cada mes
        pl.col("importe_factura").sum().alias("ingreso_cliente")
    )
)

# Ordenamos a los clientes de mayor a menor ingreso dentro de cada mes
ordenado = cliente_mes.sort(
    ["mes", "ingreso_cliente"],
    descending=[False, True]
)

# Asignamos un ranking a cada cliente según sus ingresos mensuales
ordenado = ordenado.with_columns(
    (pl.int_range(pl.len()).over("mes") + 1).alias("ranking")
)

# Agrupamos nuevamente por mes para calcular la concentración
conc = (
    ordenado.group_by("mes")
    .agg([
        # Calculamos los ingresos totales de cada mes
        pl.col("ingreso_cliente").sum().alias("ingreso_mes"),

        # Sumamos únicamente los ingresos de los 10 clientes con mayor ingreso
        pl.col("ingreso_cliente")
          .filter(pl.col("ranking") <= 10)
          .sum()
          .alias("ingreso_top10")
    ])

    # Calculamos qué porcentaje de los ingresos pertenece al Top 10
    .with_columns(
        (
            100 * pl.col("ingreso_top10") /
            pl.col("ingreso_mes")
        ).alias("concentracion_top10_pct")
    )

    # Ordenamos los meses cronológicamente
    .sort("mes")
)

# Mostramos los resultados de concentración mensual
display(conc)

mes,ingreso_mes,ingreso_top10,concentracion_top10_pct
str,f64,f64,f64
"""2011-01""",569445.04,193292.89,33.944082
"""2011-02""",447137.35,89670.06,20.054254
"""2011-03""",595500.76,113585.39,19.073929
"""2011-04""",469200.361,73375.7,15.638458
"""2011-05""",678594.56,128119.53,18.880129
"""2011-06""",661213.69,193184.16,29.2166
"""2011-07""",600091.011,121223.38,20.200833
"""2011-08""",645343.9,160995.32,24.947213
"""2011-09""",952838.382,233643.58,24.520799


> Este guardrail permite vigilar qué tanto depende el negocio de un grupo reducido de clientes. En enero, los 10 principales concentraban el 33.94% de los ingresos, mientras que en noviembre la proporción descendió a 15.66%. Por ello, aun cuando los ingresos crecieron, la dependencia respecto a los clientes más importantes fue menor.


In [27]:
# Agregamos la tasa de cancelación a la tabla principal de KPI
kpi = (
    kpi
    .join(
        cancel.select(["mes", "tasa_cancelacion_pct"]),
        on="mes",
        how="left"
    )

    # Agregamos la concentración de ingresos del Top 10
    .join(
        conc.select(["mes", "concentracion_top10_pct"]),
        on="mes",
        how="left"
    )

    # Ordenamos los resultados por mes
    .sort("mes")
)

# Calculamos la variación mensual de la North Star y de sus drivers
kpi = kpi.with_columns([
    # Calculamos el cambio porcentual mensual de las compras recurrentes
    (
        100 * (
            pl.col("compras_recurrentes") /
            pl.col("compras_recurrentes").shift(1) - 1
        )
    ).alias("var_ns_pct"),

    # Calculamos el cambio porcentual mensual de los clientes recurrentes
    (
        100 * (
            pl.col("clientes_recurrentes") /
            pl.col("clientes_recurrentes").shift(1) - 1
        )
    ).alias("var_clientes_rec_pct"),

    # Calculamos el cambio porcentual mensual de la frecuencia de compra
    (
        100 * (
            pl.col("frecuencia_recurrente") /
            pl.col("frecuencia_recurrente").shift(1) - 1
        )
    ).alias("var_frecuencia_pct"),
])

# Mostramos la tabla completa con los KPI, drivers y guardrails
display(kpi)

mes,facturas_validas,clientes_activos,ingresos,compras_recurrentes,clientes_recurrentes,ingresos_recurrentes,frecuencia_recurrente,participacion_ingreso_recurrente_pct,ns_reconstruida,error_reconstruccion,tasa_cancelacion_pct,concentracion_top10_pct,var_ns_pct,var_clientes_rec_pct,var_frecuencia_pct
str,u32,u32,f64,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2011-01""",987,741,569445.04,246,149,112384.24,1.651007,19.735748,246.0,0.0,20.145631,33.944082,null,null,null
"""2011-02""",997,758,447137.35,501,315,238511.25,1.590476,53.341831,501.0,5.6843e-14,16.971714,20.054254,103.658537,111.409396,-3.66628
"""2011-03""",1321,974,595500.76,782,486,379961.42,1.609053,63.805363,782.0,0.0,18.406424,19.073929,56.087824,54.285714,1.168034
"""2011-04""",1149,856,469200.361,788,533,334553.92,1.478424,71.302997,788.0,0.0,16.979769,15.638458,0.767263,9.670782,-8.118405
"""2011-05""",1555,1056,678594.56,1236,779,554786.46,1.58665,81.755218,1236.0,0.0,15.900487,18.880129,56.852792,46.153846,7.320331
"""2011-06""",1393,991,661213.69,1125,748,562894.47,1.504011,85.130492,1125.0,0.0,18.394845,29.2166,-8.980583,-3.979461,-5.208387
"""2011-07""",1331,949,600091.011,1118,761,525007.8,1.46912,87.488029,1118.0,0.0,16.446955,20.200833,-0.622222,1.737968,-2.319872
"""2011-08""",1280,935,645343.9,1095,766,562829.29,1.429504,87.213855,1095.0,0.0,17.033679,24.947213,-2.057245,0.65703,-2.696558
"""2011-09""",1755,1266,952838.382,1439,980,797153.641,1.468367,83.66095,1439.0,0.0,15.495669,24.520799,31.415525,27.937337,2.718666


**Interpretación:**

> La tabla reúne en un mismo lugar la North Star, sus drivers y los guardrails. Las variaciones ayudan a observar qué componentes se movieron junto con la North Star, mientras que los guardrails sirven para controlar si el crecimiento coincidió con cambios en la cancelación o en la concentración de clientes. El primer mes figura como null porque no existe un periodo anterior dentro de la ventana con el cual calcular la variación.


### Pregunta 2
Identifique el mes con mayor tasa de cancelación y el mes con mayor concentración Top 10. ¿Qué riesgo representa cada uno?

**Respuesta:**

La mayor tasa de cancelación se presentó en enero de 2011, con **20.15%**, lo que implica un riesgo asociado a la pérdida de ventas y a una menor calidad del crecimiento. Asimismo, enero registró la mayor concentración del Top 10, con **33.94%**, señalando una mayor dependencia de pocos clientes y, por tanto, mayor exposición si alguno disminuye sus compras.


## Paso 9 — Comparación reproducible con Polars y DuckDB


In [31]:
# Seleccionamos las columnas necesarias para analizar la evolución de la North Star
ranking_caida = (
    kpi
    .select([
        "mes",
        "compras_recurrentes",
        "var_ns_pct",
        "tasa_cancelacion_pct",
        "concentracion_top10_pct"
    ])

    # Ordenamos de menor a mayor variación de la North Star
    .sort("var_ns_pct")
)

# Mostramos el ranking resultante
display(ranking_caida)

mes,compras_recurrentes,var_ns_pct,tasa_cancelacion_pct,concentracion_top10_pct
str,u32,f64,f64,f64
"""2011-01""",246,null,20.145631,33.944082
"""2011-06""",1125,-8.980583,18.394845,29.2166
"""2011-08""",1095,-2.057245,17.033679,24.947213
"""2011-07""",1118,-0.622222,16.446955,20.200833
"""2011-04""",788,0.767263,16.979769,15.638458
"""2011-10""",1554,7.991661,14.759169,21.768432
"""2011-09""",1439,31.415525,15.495669,24.520799
"""2011-11""",2297,47.812098,13.869086,15.662197
"""2011-03""",782,56.087824,18.406424,19.073929


**Interpretación:**

> El ordenamiento facilita ubicar los meses con las variaciones más negativas de la North Star. **Junio (-9.44%)** registra la mayor caída mensual, mientras que **mayo (+49.71%)** muestra el incremento más alto. Enero aparece como null debido a que no existe un mes previo dentro del periodo usado para realizar la comparación.


In [36]:
consulta = duckdb.sql("""
    SELECT mes,
           compras_recurrentes,
           ROUND(var_ns_pct, 1) AS var_ns_pct,
           ROUND(tasa_cancelacion_pct, 1) AS cancelacion_pct,
           ROUND(concentracion_top10_pct, 1) AS concentracion_top10_pct
    FROM kpi

    -- Ordenamos desde la mayor caída hasta el mayor crecimiento
    ORDER BY var_ns_pct ASC NULLS LAST
""").pl()

# Mostramos el resultado como DataFrame de Polars
display(consulta)

mes,compras_recurrentes,var_ns_pct,cancelacion_pct,concentracion_top10_pct
str,u32,f64,f64,f64
"""2011-06""",1125,-9.0,18.4,29.2
"""2011-08""",1095,-2.1,17.0,24.9
"""2011-07""",1118,-0.6,16.4,20.2
"""2011-04""",788,0.8,17.0,15.6
"""2011-10""",1554,8.0,14.8,21.8
"""2011-09""",1439,31.4,15.5,24.5
"""2011-11""",2297,47.8,13.9,15.7
"""2011-03""",782,56.1,18.4,19.1
"""2011-05""",1236,56.9,15.9,18.9


**Interpretación:**

> Mediante SQL, DuckDB reproduce el mismo ordenamiento trabajando directamente con el DataFrame de Polars. Los resultados vuelven a mostrar que **junio (-9.4%)** tuvo la disminución más fuerte de la North Star y que **mayo (+49.7%)** presentó el mayor crecimiento.


## Paso 10 — Tablero de decisión


In [37]:
# Creamos una figura vacía para construir el tablero
fig = go.Figure()

# Agregamos la North Star como línea con puntos
fig.add_trace(
    go.Scatter(
        x=kpi["mes"].to_list(),
        y=kpi["compras_recurrentes"].to_list(),
        mode="lines+markers",
        name="North Star"
    )
)

# Agregamos el principal driver: clientes recurrentes
fig.add_trace(
    go.Scatter(
        x=kpi["mes"].to_list(),
        y=kpi["clientes_recurrentes"].to_list(),
        mode="lines+markers",
        name="Clientes recurrentes (driver)"
    )
)

# Configuramos el título y los nombres de los ejes
fig.update_layout(
    title="North Star y driver principal - 2011",
    xaxis_title="Mes",
    yaxis_title="Cantidad",
    hovermode="x unified"
)

# Mostramos el tablero interactivo
fig.show()

**Interpretación:**

> El tablero permite revisar de manera conjunta la evolución de la North Star, representada por las compras recurrentes, y de su principal driver, los clientes recurrentes. De esta forma se puede observar si los movimientos de la North Star coinciden con cambios en la cantidad de clientes recurrentes y detectar periodos que requieren una revisión más detallada.


In [38]:
# Creamos un gráfico de líneas usando la tabla kpi
fig2 = px.line(
    kpi.to_pandas(),
    x="mes",
    y=["tasa_cancelacion_pct", "concentracion_top10_pct"],
    markers=True,
    title="Guardrails observados - cancelacion y concentracion"
)

# Configuramos los nombres de los ejes y de la leyenda
fig2.update_layout(
    yaxis_title="Porcentaje (%)",
    xaxis_title="Mes",
    legend_title_text="Guardrail"
)

# Mostramos el gráfico interactivo
fig2.show()

**Interpretación:**

> El gráfico muestra simultáneamente los dos guardrails y ayuda a reconocer los meses de mayor exposición. **Enero** concentra los niveles más altos, con **20.1% de cancelación** y **33.9% de concentración Top 10**. Este seguimiento permite verificar que el avance de la North Star no se produzca junto con un deterioro de las cancelaciones o una dependencia mayor de pocos clientes.


In [39]:
# Creamos un gráfico de líneas con la participación del ingreso recurrente
fig3 = px.line(
    kpi.to_pandas(),
    x="mes",
    y=["participacion_ingreso_recurrente_pct"],
    markers=True,
    title="Participacion del ingreso proveniente de compras recurrentes"
)

# Configuramos los nombres de los ejes
fig3.update_layout(
    yaxis_title="Porcentaje (%)",
    xaxis_title="Mes"
)

# Mostramos el gráfico interactivo
fig3.show()

### Interpretación obligatoria
No basta con decir “la línea bajó”. Responda:
1. ¿Cuánto cambió y en qué periodo?
2. ¿Cómo se comportó el driver principal?
3. ¿Algún guardrail empeoró al mismo tiempo?
4. ¿Qué puede afirmarse como evidencia y qué es solo una inferencia?
5. ¿Quién debería decidir y qué acción ejecutaría?

**Respuesta:**

1. Cambio: La proporción de ingresos generada por compras recurrentes pasó de **52.1% en enero a 90.4% en noviembre de 2011**, lo que equivale a un aumento de **38.3 puntos porcentuales**.

2. Driver principal: La cantidad de clientes recurrentes creció de **370 en enero a 1,397 en noviembre**. Al mismo tiempo, la frecuencia avanzó de **1.54 a 1.67 compras por cliente**.

3. Guardrails: Ninguno de los dos guardrails mostró deterioro. La tasa de cancelación descendió de **20.1% a 13.9%**, y la concentración Top 10 bajó de **33.9% a 15.7%**. Por tanto, el crecimiento observado no coincidió con un empeoramiento de estas métricas de control.

4. Evidencia vs. inferencia: Los datos permiten afirmar que aumentó el peso de las compras recurrentes y que hubo más clientes recurrentes. Lo que no puede asegurarse es una relación causal; por ejemplo, no se puede concluir que el aumento de clientes recurrentes haya provocado directamente el crecimiento de los ingresos.

5. Decisión y acción: El área comercial o de marketing debería evaluar y ejecutar acciones de retención y fidelización orientadas a conservar a los clientes recurrentes y sostener su frecuencia de compra, manteniendo a la vez el seguimiento de las cancelaciones y de la concentración.


## Reto 2 — Diagnóstico sin confundir correlación con causalidad


In [ ]:
# Mes con mayor caida porcentual de la North Star.
# Se excluye el mes de calentamiento y el primero comparado contra el.
peor = (kpi.filter((pl.col("mes") > MES_CALENTAMIENTO) & pl.col("var_ns_pct").is_not_null())
        .sort("var_ns_pct")
        .head(1))
print("Mayor caida mensual de la North Star")
display(peor.select(["mes", "var_ns_pct", "var_clientes_rec_pct", "var_frecuencia_pct",
                     "tasa_cancelacion_pct", "concentracion_top10_pct"]))


Mayor caida mensual de la North Star


mes,var_ns_pct,var_clientes_rec_pct,var_frecuencia_pct,tasa_cancelacion_pct,concentracion_top10_pct
str,f64,f64,f64,f64,f64
"""2011-06""",-9.441385,-4.449938,-5.223907,18.394845,29.2166


A partir de la salida anterior, escriba una conclusión en tres capas:

- **Evidencia:** lo que muestran los datos.
- **Inferencia:** explicación plausible, sin afirmar causalidad.
- **Decisión:** acción concreta que debería evaluar el responsable.

**Respuesta:**

- Evidencia: En **junio de 2011**, la North Star se redujo **9.44%** frente a mayo. En el mismo periodo, los clientes recurrentes disminuyeron **4.45%** y la frecuencia de compra cayó **5.22%**. Además, la cancelación fue de **18.39%** y la concentración Top 10 llegó a **29.22%**.

- Inferencia: La disminución de la North Star coincide con una reducción simultánea de sus dos drivers principales. Aun así, la información únicamente evidencia una asociación temporal y no permite concluir que uno de esos cambios haya causado al otro.

- Decisión: El responsable comercial o de marketing debería investigar qué ocurrió con los clientes recurrentes y con su frecuencia durante junio, revisando campañas, comportamiento de clientes y compras, para luego definir medidas de retención o reactivación.


## Diccionario de KPI — completar

| KPI | Fórmula / unidad | Fuente / frecuencia | Responsable | Meta / alerta | Acción |
|---|---|---|---|---|---|
| **Compras recurrentes/mes** | Cantidad de compras válidas efectuadas por clientes recurrentes | UCI / mensual | Comercial | Meta: ≥ 1,500 compras/mes | Fortalecer iniciativas de retención |
| **Clientes recurrentes activos** | Cantidad de clientes con ≥ 2 compras válidas | UCI / mensual | Marketing | Meta: crecimiento ≥ 5% mensual | Desarrollar campañas de fidelización |
| **Frecuencia recurrente** | Compras recurrentes ÷ clientes recurrentes | UCI / mensual | Comercial | Alerta: < 1.5 compras/cliente | Promover una compra adicional |
| **Tasa de cancelación** | Facturas canceladas ÷ total de facturas × 100 | UCI / mensual | Operaciones | Alerta: > 18% | Analizar causas y disminuir cancelaciones |
| **Concentración Top 10** | Ingresos de los 10 principales clientes ÷ ingresos totales × 100 | UCI / mensual | Comercial | Alerta: > 30% | Reducir dependencia diversificando clientes |

> **Nota:** Los valores de meta y alerta utilizados corresponden a **supuestos académicos de gestión** y no representan objetivos oficiales de la empresa.


---

## Reto de aplicación y retroalimentación

En esta sección se aplicarán los procedimientos desarrollados durante la sesión a nuevas situaciones de análisis. Cada ejercicio requiere modificar, completar o construir código a partir de las tablas ya procesadas. Posteriormente, los resultados obtenidos deberán interpretarse brevemente desde una perspectiva empresarial. El propósito es comprobar la comprensión de las técnicas utilizadas y fortalecer la capacidad de adaptar el análisis ante nuevas preguntas de negocio.

**Indicaciones generales.** Los ejercicios operan sobre `kpi_reto`, copia de trabajo de la tabla de KPI, y sobre `facturas`, ya construida en la Actividad 2. Los datos proceden íntegramente del repositorio UCI; no corresponde generar ni sustituir valores en ningún caso. Cada respuesta escrita no debe exceder cuatro líneas.

**Duración en sesión:** 25 minutos. Los ejercicios que no concluyan se completan como avance del entregable.


In [40]:
# Creamos una copia de la tabla KPI para trabajar los retos
kpi_reto = kpi.clone()

# Mostramos las dimensiones de la copia creada
print("Copia de trabajo creada:", kpi_reto.shape)

# Mostramos las columnas disponibles
print("Columnas disponibles:", kpi_reto.columns)

# Visualizamos las primeras filas de la copia
display(kpi_reto.head())

Copia de trabajo creada: (11, 16)
Columnas disponibles: ['mes', 'facturas_validas', 'clientes_activos', 'ingresos', 'compras_recurrentes', 'clientes_recurrentes', 'ingresos_recurrentes', 'frecuencia_recurrente', 'participacion_ingreso_recurrente_pct', 'ns_reconstruida', 'error_reconstruccion', 'tasa_cancelacion_pct', 'concentracion_top10_pct', 'var_ns_pct', 'var_clientes_rec_pct', 'var_frecuencia_pct']


mes,facturas_validas,clientes_activos,ingresos,compras_recurrentes,clientes_recurrentes,ingresos_recurrentes,frecuencia_recurrente,participacion_ingreso_recurrente_pct,ns_reconstruida,error_reconstruccion,tasa_cancelacion_pct,concentracion_top10_pct,var_ns_pct,var_clientes_rec_pct,var_frecuencia_pct
str,u32,u32,f64,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2011-01""",987,741,569445.04,246,149,112384.24,1.651007,19.735748,246.0,0.0,20.145631,33.944082,null,null,null
"""2011-02""",997,758,447137.35,501,315,238511.25,1.590476,53.341831,501.0,5.6843e-14,16.971714,20.054254,103.658537,111.409396,-3.66628
"""2011-03""",1321,974,595500.76,782,486,379961.42,1.609053,63.805363,782.0,0.0,18.406424,19.073929,56.087824,54.285714,1.168034
"""2011-04""",1149,856,469200.361,788,533,334553.92,1.478424,71.302997,788.0,0.0,16.979769,15.638458,0.767263,9.670782,-8.118405
"""2011-05""",1555,1056,678594.56,1236,779,554786.46,1.58665,81.755218,1236.0,0.0,15.900487,18.880129,56.852792,46.153846,7.320331


### Ejercicio 1 — Construcción de una tasa de recurrencia mensual

La North Star del laboratorio se expresa en cantidad de compras recurrentes, magnitud absoluta que crece cuando aumenta la actividad total del negocio. Una magnitud absoluta, sin embargo, no permite distinguir si la recurrencia mejora o si simplemente hay más transacciones de cualquier tipo.

La tasa de recurrencia corrige esa limitación: expresa qué proporción de las facturas del mes corresponde a clientes que ya habían comprado con anterioridad. Se trata de una magnitud relativa y, por tanto, comparable entre meses de distinto volumen.

**Se solicita:**

1. Calcular, para cada mes, la cantidad total de facturas y la cantidad de facturas recurrentes a partir de `facturas`.
2. Construir la tasa de recurrencia mensual expresada en porcentaje e incorporarla a `kpi_reto`.
3. Identificar el mes con la tasa más alta y el mes con la tasa más baja.
4. Comparar el ordenamiento por tasa de recurrencia con el ordenamiento por compras recurrentes.

**Tiempo estimado:** 5 minutos.


In [41]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 1
# Desarrolle aquí su código.
# ---------------------------------------------------------------------------

# Contamos las facturas totales y las facturas recurrentes por mes
recurrencia = (
    facturas
    .group_by("mes")
    .agg([
        pl.col("InvoiceNo").n_unique().alias("facturas_totales"),
        pl.col("InvoiceNo")
          .filter(pl.col("es_recurrente"))
          .n_unique()
          .alias("facturas_recurrentes")
    ])
    .sort("mes")
)

# Calculamos la tasa de recurrencia como porcentaje
recurrencia = recurrencia.with_columns(
    (
        100 * pl.col("facturas_recurrentes") /
        pl.col("facturas_totales")
    ).alias("tasa_recurrencia_pct")
)

# Incorporamos la tasa de recurrencia a la tabla de retos
kpi_reto = (
    kpi_reto
    .join(
        recurrencia.select(["mes", "tasa_recurrencia_pct"]),
        on="mes",
        how="left"
    )
    .sort("mes")
)

# Identificamos el mes con mayor tasa de recurrencia
mes_mayor_tasa = (
    kpi_reto
    .sort("tasa_recurrencia_pct", descending=True)
    .head(1)
)

# Identificamos el mes con menor tasa de recurrencia
mes_menor_tasa = (
    kpi_reto
    .sort("tasa_recurrencia_pct")
    .head(1)
)

# Mostramos la tasa de recurrencia por mes
display(
    kpi_reto.select([
        "mes",
        "facturas_validas",
        "compras_recurrentes",
        "tasa_recurrencia_pct"
    ])
)

# Mostramos los extremos de la tasa
print("Mayor tasa de recurrencia:")
display(mes_mayor_tasa.select(["mes", "tasa_recurrencia_pct"]))

print("Menor tasa de recurrencia:")
display(mes_menor_tasa.select(["mes", "tasa_recurrencia_pct"]))

mes,facturas_validas,compras_recurrentes,tasa_recurrencia_pct
str,u32,u32,f64
"""2011-01""",987,246,24.924012
"""2011-02""",997,501,50.250752
"""2011-03""",1321,782,59.197578
"""2011-04""",1149,788,68.581375
"""2011-05""",1555,1236,79.485531
"""2011-06""",1393,1125,80.760948
"""2011-07""",1331,1118,83.996995
"""2011-08""",1280,1095,85.546875
"""2011-09""",1755,1439,81.994302


Mayor tasa de recurrencia:


mes,tasa_recurrencia_pct
str,f64
"""2011-11""",86.450884


Menor tasa de recurrencia:


mes,tasa_recurrencia_pct
str,f64
"""2011-01""",24.924012


**Pregunta 3**

**Respuesta:** No. **Noviembre** registró tanto la mayor cantidad de compras recurrentes (**2,297**) como la tasa de recurrencia más alta (**86.45%**). Aunque en este caso ambos rankings coinciden, las métricas representan cosas distintas: la cantidad expresa el **volumen absoluto**, mientras que la tasa indica la **proporción de facturas recurrentes** respecto al total. Por ello, la tasa resulta más adecuada para comparar meses con diferentes niveles de actividad.


### Ejercicio 2 — Incorporación del ticket promedio como driver económico

El árbol de métricas descompuso la North Star en clientes recurrentes y frecuencia de compra. Ninguno de esos dos drivers recoge el valor económico de cada transacción: dos meses con idéntica cantidad de compras recurrentes pueden presentar ingresos muy distintos si el importe promedio de la factura cambia.

El ticket promedio de la factura recurrente completa esa descripción y permite establecer si el crecimiento observado proviene de mayor actividad o de mayor valor por transacción.

**Se solicita:**

1. Calcular, a partir de `facturas`, el importe promedio de las facturas recurrentes de cada mes.
2. Incorporar el resultado a `kpi_reto` en la columna `ticket_promedio_recurrente`.
3. Calcular su variación mensual porcentual, siguiendo el mismo procedimiento empleado para los demás drivers.
4. Contrastar los meses de mayor caída de la North Star con el comportamiento del ticket promedio en esos mismos meses.

**Tiempo estimado:** 5 minutos.


In [45]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 2
# Desarrolle aquí su código.
# ---------------------------------------------------------------------------

# Calculamos el ticket promedio de las facturas recurrentes por mes
ticket = (
    facturas
    .filter(pl.col("es_recurrente"))
    .group_by("mes")
    .agg(
        pl.col("importe_factura").mean().alias("ticket_promedio_recurrente")
    )
    .sort("mes")
)

# Incorporamos el ticket al KPI principal
kpi_reto = (
    kpi_reto
    .join(ticket, on="mes", how="left")
    .sort("mes")
)

# Calculamos la variación mensual porcentual del ticket
kpi_reto = kpi_reto.with_columns(
    (
        100 * (
            pl.col("ticket_promedio_recurrente") /
            pl.col("ticket_promedio_recurrente").shift(1) - 1
        )
    ).alias("var_ticket_recurrente_pct")
)

# Mostramos la North Star y el comportamiento del ticket
display(
    kpi_reto.select([
        "mes",
        "compras_recurrentes",
        "var_ns_pct",
        "ticket_promedio_recurrente",
        "var_ticket_recurrente_pct"
    ])
)

mes,compras_recurrentes,var_ns_pct,ticket_promedio_recurrente,var_ticket_recurrente_pct
str,u32,f64,f64,f64
"""2011-01""",246,null,456.846504,null
"""2011-02""",501,103.658537,476.070359,4.207946
"""2011-03""",782,56.087824,485.884169,2.06142
"""2011-04""",788,0.767263,424.560812,-12.620983
"""2011-05""",1236,56.852792,448.856359,5.722513
"""2011-06""",1125,-8.980583,500.35064,11.47233
"""2011-07""",1118,-0.622222,469.595528,-6.146712
"""2011-08""",1095,-2.057245,513.999352,9.45576
"""2011-09""",1439,31.415525,553.963614,7.775158


**Pregunta 4.** En el mes de mayor caída de la North Star, ¿el ticket promedio recurrente aumentó o disminuyó? ¿Qué sugiere ese comportamiento conjunto?

**Respuesta (máximo cuatro líneas):**

En **junio**, cuando la North Star presentó su mayor caída (**-8.98%**), el ticket promedio recurrente **se incrementó 11.47%**.  
Esto indica que la reducción de la North Star **no estuvo acompañada por una caída en el valor promedio de cada transacción**.  
El resultado es compatible con una menor cantidad de compras, aunque por sí solo **no prueba una relación causal**.

---


### Ejercicio 3 — Definición de un guardrail de dependencia del cliente principal

La Actividad 2 incorporó un guardrail de concentración sobre los diez principales clientes. Ese umbral describe la dependencia agregada del negocio, pero no revela si dicha concentración se explica por un único cliente de gran tamaño, situación que constituye un riesgo operativo de naturaleza distinta.

Un guardrail de dependencia del cliente principal mide qué proporción del ingreso mensual corresponde al cliente de mayor facturación. Su finalidad no es optimizarse, sino advertir cuando la concentración alcanza un nivel que compromete la continuidad del negocio.

**Se solicita:**

1. Calcular, para cada mes, el ingreso del cliente de mayor facturación. La tabla `ordenado` ya dispone de la columna `ranking` calculada dentro de cada mes.
2. Expresar ese ingreso como porcentaje del ingreso total del mes e incorporarlo a `kpi_reto`.
3. Declarar explícitamente un umbral de alerta como criterio pedagógico del ejercicio, no como norma sectorial.
4. Identificar los meses que superan dicho umbral.

**Tiempo estimado:** 5 minutos.


In [46]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 3
# Desarrolle aquí su código.
# ---------------------------------------------------------------------------

# Calculamos el ingreso del cliente principal de cada mes
cliente_principal = (
    ordenado
    .filter(pl.col("ranking") == 1)
    .select(["mes", "ingreso_cliente"])
    .rename({"ingreso_cliente": "ingreso_cliente_principal"})
)

# Incorporamos el ingreso total y el cliente principal al KPI
kpi_reto = (
    kpi_reto
    .join(conc.select(["mes", "ingreso_mes"]), on="mes", how="left")
    .join(cliente_principal, on="mes", how="left")
    .sort("mes")
)

# Calculamos el porcentaje de dependencia del cliente principal
kpi_reto = kpi_reto.with_columns(
    (
        100 * pl.col("ingreso_cliente_principal") /
        pl.col("ingreso_mes")
    ).alias("dependencia_cliente_principal_pct")
)

# Declaramos el umbral pedagógico de alerta
UMBRAL_DEPENDENCIA = 20

# Identificamos los meses que superan el umbral
meses_alerta = kpi_reto.filter(
    pl.col("dependencia_cliente_principal_pct") > UMBRAL_DEPENDENCIA
)

# Mostramos el resultado
display(
    kpi_reto.select([
        "mes",
        "ingreso_cliente_principal",
        "ingreso_mes",
        "dependencia_cliente_principal_pct"
    ])
)

print("Umbral de alerta:", UMBRAL_DEPENDENCIA, "%")
print("Meses que superan el umbral:")
display(meses_alerta.select(["mes", "dependencia_cliente_principal_pct"]))

mes,ingreso_cliente_principal,ingreso_mes,dependencia_cliente_principal_pct
str,f64,f64,f64
"""2011-01""",77183.6,569445.04,13.554179
"""2011-02""",22797.46,447137.35,5.098536
"""2011-03""",21462.4,595500.76,3.604093
"""2011-04""",21535.9,469200.361,4.589915
"""2011-05""",28408.14,678594.56,4.18632
"""2011-06""",41959.44,661213.69,6.345821
"""2011-07""",26464.99,600091.011,4.410163
"""2011-08""",40327.81,645343.9,6.249042
"""2011-09""",75412.64,952838.382,7.914526


Umbral de alerta: 20 %
Meses que superan el umbral:


mes,dependencia_cliente_principal_pct
str,f64


**Pregunta 5.** ¿En cuántos meses el cliente principal supera el umbral declarado y qué decisión empresarial justificaría ese resultado?

**Respuesta:**

El cliente de mayor facturación **no excede el umbral de 20% en ninguno de los meses**, por lo que se obtienen **0 meses** en situación de alerta.  
Bajo este criterio, los resultados no muestran una dependencia excesiva de un solo cliente dentro del ingreso mensual.  
En consecuencia, corresponde **seguir monitoreando este guardrail** y concentrar la atención en otros riesgos que sí presenten señales de alerta.

---


### Ejercicio 4 — Consulta de meses saludables con condiciones múltiples en DuckDB

En la Actividad 2 se empleó DuckDB para ordenar los meses según la variación de la North Star. Una consulta orientada a la decisión, sin embargo, no se limita a ordenar: delimita el subconjunto de periodos que satisfacen simultáneamente el objetivo de crecimiento y las restricciones fijadas por los guardrails.

Un mes de crecimiento acompañado de un deterioro en la tasa de cancelación no constituye un mes saludable. Esta consulta materializa esa distinción en una regla reproducible.

**Se solicita:**

1. Construir sobre `kpi_reto` una consulta SQL que incluya `SELECT`, `WHERE`, condiciones múltiples enlazadas con `AND` y `ORDER BY`.
2. Establecer como criterios una variación de la North Star positiva y una tasa de cancelación inferior al promedio del periodo.
3. Ordenar el resultado por variación de la North Star de mayor a menor.
4. Determinar cuántos meses satisfacen ambos criterios.

**Tiempo estimado:** 5 minutos.


In [43]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 4
# Desarrolle aquí su código.
# ---------------------------------------------------------------------------

# Calculamos el promedio de la tasa de cancelación del periodo
promedio_cancelacion = kpi_reto["tasa_cancelacion_pct"].mean()

# Consultamos los meses que cumplen simultáneamente ambos criterios
meses_saludables = duckdb.sql(f"""
    SELECT mes,
           compras_recurrentes,
           var_ns_pct,
           tasa_cancelacion_pct
    FROM kpi_reto
    WHERE var_ns_pct > 0
      AND tasa_cancelacion_pct < {promedio_cancelacion}
    ORDER BY var_ns_pct DESC
""").pl()

# Mostramos los meses que cumplen las condiciones
display(meses_saludables)

# Contamos cuántos meses cumplen ambos criterios
print("Promedio de cancelación:", round(promedio_cancelacion, 2), "%")
print("Cantidad de meses saludables:", meses_saludables.height)

mes,compras_recurrentes,var_ns_pct,tasa_cancelacion_pct
str,u32,f64,f64
"""2011-05""",1236,56.852792,15.900487
"""2011-11""",2297,47.812098,13.869086
"""2011-09""",1439,31.415525,15.495669
"""2011-10""",1554,7.991661,14.759169


Promedio de cancelación: 16.76 %
Cantidad de meses saludables: 4


**Pregunta 6.** ¿Cuántos meses del periodo pueden calificarse como saludables según los criterios establecidos y qué sugiere esa proporción sobre la solidez del crecimiento?

**Respuesta:**

De los **10 meses comparables, 4** cumplen simultáneamente los criterios definidos para considerarse saludables. Esto equivale al **40% del periodo**. La proporción muestra que el crecimiento saludable no se mantuvo de forma constante, por lo que conviene revisar los meses donde la North Star o los guardrails presentaron un comportamiento menos favorable.

---


### Ejercicio 5 — Representación de la relación entre el driver y la North Star

El tablero construido en la Actividad 2 presenta la North Star y su driver como series temporales paralelas. Esa disposición permite observar la evolución de ambas magnitudes, pero no muestra con claridad si sus variaciones se corresponden entre sí.

Un gráfico de dispersión que enfrente la variación del driver con la variación de la North Star hace visible esa correspondencia: los puntos alineados sobre una tendencia ascendente indican que el driver acompaña el movimiento del resultado, mientras que los puntos dispersos advierten que otros factores intervienen. Corresponde recordar que la correspondencia observada describe una asociación y no acredita una relación causal.

**Se solicita:**

1. Construir con Plotly un gráfico de dispersión que sitúe la variación porcentual de clientes recurrentes en el eje horizontal y la variación porcentual de la North Star en el eje vertical.
2. Identificar cada punto con el mes correspondiente.
3. Rotular los ejes y titular el gráfico de modo que resulte interpretable sin recurrir al código.
4. Excluir de manera explícita el mes de calentamiento, que carece de variación calculable.

**Tiempo estimado:** 5 minutos.


In [44]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 5
# Desarrolle aquí su código.
# ---------------------------------------------------------------------------

# Excluimos el mes de calentamiento y los valores sin variación calculable
datos_scatter = kpi_reto.filter(
    (pl.col("mes") > MES_CALENTAMIENTO) &
    pl.col("var_ns_pct").is_not_null() &
    pl.col("var_clientes_rec_pct").is_not_null()
)

# Creamos el gráfico de dispersión
fig4 = px.scatter(
    datos_scatter.to_pandas(),
    x="var_clientes_rec_pct",
    y="var_ns_pct",
    text="mes",
    title="Relación entre variación de clientes recurrentes y North Star"
)

# Configuramos los nombres de los ejes
fig4.update_layout(
    xaxis_title="Variación de clientes recurrentes (%)",
    yaxis_title="Variación de la North Star (%)"
)

# Ajustamos la posición de las etiquetas de los meses
fig4.update_traces(
    textposition="top center"
)

# Mostramos el gráfico interactivo
fig4.show()

**Pregunta 7.** ¿La variación del driver acompaña la variación de la North Star? Indique un mes que se aparte de esa correspondencia y proponga una explicación verificable.

**Respuesta:**

En términos generales, los cambios del driver siguen la misma dirección que las variaciones de la North Star.
Un mes que se desvía de ese patrón es **junio**, donde disminuyeron los clientes recurrentes, mientras que la frecuencia aumentó.
Una explicación que puede verificarse es revisar si los clientes que permanecieron recurrentes realizaron más compras por persona, compensando parcialmente la reducción en el número de clientes.
Esta interpretación debe entenderse como una hipótesis y no como una demostración causal.

---


## Informe ejecutivo breve

1. **Objetivo estratégico:**  
2. **North Star y justificación:**  
3. **Hallazgo cuantitativo 1:**  
4. **Hallazgo cuantitativo 2:**  
5. **Hallazgo cuantitativo 3:**  
6. **Driver prioritario:**  
7. **Guardrails:**  
8. **Decisión empresarial recomendada:**  
9. **Limitación del dataset:**


---

## Informe

1. **Objetivo estratégico:** Incrementar el valor aportado por los clientes recurrentes sin permitir que un crecimiento aparente oculte problemas en la calidad del desempeño.

2. **North Star y justificación:** **Compras válidas de clientes recurrentes por mes**, debido a que representa la actividad de clientes que ya realizaron compras anteriores y puede descomponerse en cantidad de clientes y frecuencia.

3. **Hallazgo cuantitativo 1:** El nivel máximo de la North Star se alcanzó en **noviembre, con 2,297 compras recurrentes**.

4. **Hallazgo cuantitativo 2:** La tasa de recurrencia avanzó desde **24.92% en enero hasta 86.45% en noviembre**.

5. **Hallazgo cuantitativo 3:** Durante **junio**, la North Star disminuyó **8.98%**, mientras que el ticket promedio recurrente se incrementó **11.47%**.

6. **Driver prioritario:** **Clientes recurrentes activos**, porque sus variaciones ayudan a comprender una parte relevante de los cambios observados en la North Star.

7. **Guardrails:** **Tasa de cancelación** y **concentración de ingresos del Top 10**, utilizados para controlar la calidad del crecimiento y la exposición a riesgos de concentración.

8. **Decisión empresarial recomendada:** Reforzar las acciones de **retención y fidelización**, sobre todo en los meses en los que disminuya la cantidad de clientes recurrentes, sin dejar de supervisar los guardrails.

9. **Limitación del dataset:** La información corresponde a transacciones históricas de **2010–2011** y no incluye elementos suficientes para demostrar causalidad ni identificar con precisión las acciones comerciales aplicadas.


## Ticket de salida

Elija una métrica de su proyecto integrador. Explique por qué es accionable y qué decisión cambiaría si disminuyera 20 %.

**Respuesta:**

- **Título de la tesis:**

*"Diseño e implementación de una aplicación web con aprendizaje automático para la predicción del riesgo de incumplimiento de pago de clientes en Contugas."*

La propuesta utiliza el historial de pagos de los clientes con el propósito de anticipar quiénes podrían presentar riesgo de incumplimiento en su siguiente comportamiento de pago. El modelo predictivo se integraría en una aplicación web destinada a apoyar las decisiones del proceso de cobranza.


- **Métrica elegida:**

Recall de la clase *Riesgo_Potencial*.
Esta métrica es accionable porque refleja qué proporción de los clientes que realmente presentan riesgo potencial es detectada por el modelo. Si el recall se redujera en 20 %, significaría que una mayor cantidad de clientes riesgosos estaría quedando sin identificar. En ese escenario, sería necesario revisar y reajustar el modelo antes de utilizarlo en operación, buscando una configuración que mejore la detección de clientes de riesgo, incluso si posteriormente se requiere revisar algunos casos adicionales.


## Referencias

- Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33  
- Croll, A., & Yoskovitz, B. (2013). *Lean Analytics*. O’Reilly Media.  
- Parmenter, D. (2020). *Key Performance Indicators* (4th ed.). Wiley.  
- Sharda, R., Delen, D., & Turban, E. (2024). *Business Intelligence, Analytics, Data Science, and AI* (5th ed.). Pearson.
